In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wkirgsn/electric-motor-temperature")

print("Path to dataset files:", path)


Path to dataset files: /kaggle/input/datasets/wkirgsn/electric-motor-temperature


In [13]:
import os
import pandas as pd
    
df = pd.read_csv('/kaggle/input/datasets/wkirgsn/electric-motor-temperature/measures_v2.csv', nrows=5)

print(df.columns.tolist())

['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'profile_id']


In [14]:
# ==========================================================
# STEP 1 - LOAD AND VALIDATE PMSM DATASET
# ==========================================================

import pandas as pd
import numpy as np

# -----------------------------
# Dataset Path
# -----------------------------
DATA_PATH = "/kaggle/input/datasets/wkirgsn/electric-motor-temperature/measures_v2.csv"

print("=" * 60)
print("Loading PMSM Dataset...")
print("=" * 60)

# Load using float32 to reduce memory
df = pd.read_csv(DATA_PATH, dtype=np.float32)

print("\nDataset Loaded Successfully.\n")

# --------------------------------------------------
# Basic Information
# --------------------------------------------------
print("=" * 60)
print("Dataset Shape")
print("=" * 60)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

# --------------------------------------------------
# Column Names
# --------------------------------------------------
print("\nColumns")

print(df.columns.tolist())

# --------------------------------------------------
# Data Types
# --------------------------------------------------
print("\nData Types")

print(df.dtypes)

# --------------------------------------------------
# Missing Values
# --------------------------------------------------
print("\nMissing Values")

missing = df.isnull().sum()

print(missing)

print(f"\nTotal Missing Values : {missing.sum():,}")

# --------------------------------------------------
# Duplicate Rows
# --------------------------------------------------
duplicates = df.duplicated().sum()

print(f"\nDuplicate Rows : {duplicates:,}")

# --------------------------------------------------
# Memory Usage
# --------------------------------------------------
memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"\nMemory Usage : {memory_mb:.2f} MB")

# --------------------------------------------------
# Statistical Summary
# --------------------------------------------------
print("\nSummary Statistics")

print(df.describe().T)

# --------------------------------------------------
# Verify Required Columns
# --------------------------------------------------

required_columns = [
    "u_q",
    "coolant",
    "stator_winding",
    "u_d",
    "stator_tooth",
    "motor_speed",
    "i_d",
    "i_q",
    "pm",
    "stator_yoke",
    "ambient",
    "torque",
    "profile_id"
]

missing_columns = list(set(required_columns) - set(df.columns))

if len(missing_columns) == 0:
    print("\n✅ Dataset validation successful.")
else:
    print("\n❌ Missing columns:")
    print(missing_columns)

print("\nStep 1 Completed Successfully.")

Loading PMSM Dataset...

Dataset Loaded Successfully.

Dataset Shape
Rows    : 1,330,816
Columns : 13

Columns
['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'profile_id']

Data Types
u_q               float32
coolant           float32
stator_winding    float32
u_d               float32
stator_tooth      float32
motor_speed       float32
i_d               float32
i_q               float32
pm                float32
stator_yoke       float32
ambient           float32
torque            float32
profile_id        float32
dtype: object

Missing Values
u_q               0
coolant           0
stator_winding    0
u_d               0
stator_tooth      0
motor_speed       0
i_d               0
i_q               0
pm                0
stator_yoke       0
ambient           0
torque            0
profile_id        0
dtype: int64

Total Missing Values : 0

Duplicate Rows : 0

Memory Usage : 66.00 MB

Summary Statistics


In [15]:
# ==========================================================
# STEP 2 - PHYSICS-AWARE DATA CLEANING
# ==========================================================

import numpy as np
import pandas as pd

print("="*60)
print("STEP 2 : DATA CLEANING")
print("="*60)

# --------------------------------------------------------
# Initial shape
# --------------------------------------------------------

print(f"\nInitial Shape : {df.shape}")

# --------------------------------------------------------
# Remove Duplicate Rows
# --------------------------------------------------------

duplicate_rows = df.duplicated().sum()

print(f"\nDuplicate Rows Found : {duplicate_rows}")

if duplicate_rows > 0:
    df = df.drop_duplicates()

print(f"Shape After Duplicate Removal : {df.shape}")

# --------------------------------------------------------
# Replace Infinite Values
# --------------------------------------------------------

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# --------------------------------------------------------
# Remove Missing Values
# --------------------------------------------------------

missing_before = df.isna().sum().sum()

print(f"\nMissing Values Before Cleaning : {missing_before}")

df.dropna(inplace=True)

missing_after = df.isna().sum().sum()

print(f"Missing Values After Cleaning : {missing_after}")

print(f"Shape After NaN Removal : {df.shape}")

# --------------------------------------------------------
# Sort by Driving Profile
# --------------------------------------------------------

df = df.sort_values(by="profile_id").reset_index(drop=True)

print("\nDataset Sorted by profile_id")

# --------------------------------------------------------
# Physical Range Checks
# --------------------------------------------------------

print("\nChecking Physical Ranges")

physical_limits = {
    "motor_speed": (-20000, 20000),
    "u_d": (-500, 500),
    "u_q": (-500, 500),
    "i_d": (-1000, 1000),
    "i_q": (-1000, 1000),
    "torque": (-500, 500),
    "ambient": (-50, 100),
    "coolant": (-50, 150),
    "pm": (-50, 250),
    "stator_yoke": (-50, 250),
    "stator_tooth": (-50, 250),
    "stator_winding": (-50, 250)
}

for column, (low, high) in physical_limits.items():

    outside = ((df[column] < low) | (df[column] > high)).sum()

    print(f"{column:18s} -> {outside:8d} suspicious values")

print("\nCleaning Completed.")

print(f"\nCurrent Dataset Shape : {df.shape}")

STEP 2 : DATA CLEANING

Initial Shape : (1330816, 13)

Duplicate Rows Found : 0
Shape After Duplicate Removal : (1330816, 13)

Missing Values Before Cleaning : 0
Missing Values After Cleaning : 0
Shape After NaN Removal : (1330816, 13)

Dataset Sorted by profile_id

Checking Physical Ranges
motor_speed        ->        0 suspicious values
u_d                ->        0 suspicious values
u_q                ->        0 suspicious values
i_d                ->        0 suspicious values
i_q                ->        0 suspicious values
torque             ->        0 suspicious values
ambient            ->        0 suspicious values
coolant            ->        0 suspicious values
pm                 ->        0 suspicious values
stator_yoke        ->        0 suspicious values
stator_tooth       ->        0 suspicious values
stator_winding     ->        0 suspicious values

Cleaning Completed.

Current Dataset Shape : (1330816, 13)


In [16]:
# ==========================================================
# STEP 3 - PHYSICS-BASED FEATURE ENGINEERING
# ==========================================================

import numpy as np

print("=" * 60)
print("STEP 3 : PHYSICS FEATURE ENGINEERING")
print("=" * 60)

# --------------------------------------------------------
# Motor Parameters
# --------------------------------------------------------

Rs = 0.05        # Nominal stator resistance (Ohm)

# --------------------------------------------------------
# Current Magnitude
# --------------------------------------------------------

df["current_mag"] = np.sqrt(df["i_d"]**2 + df["i_q"]**2)

# --------------------------------------------------------
# Voltage Magnitude
# --------------------------------------------------------

df["voltage_mag"] = np.sqrt(df["u_d"]**2 + df["u_q"]**2)

# --------------------------------------------------------
# Electrical Input Power
# --------------------------------------------------------

df["electrical_power"] = (
    df["u_d"] * df["i_d"] +
    df["u_q"] * df["i_q"]
)

# --------------------------------------------------------
# Apparent Power
# --------------------------------------------------------

df["apparent_power"] = (
    df["voltage_mag"] *
    df["current_mag"]
)

# --------------------------------------------------------
# Copper Loss
# --------------------------------------------------------

df["copper_loss"] = (
    1.5 *
    Rs *
    (
        df["i_d"]**2 +
        df["i_q"]**2
    )
)

# --------------------------------------------------------
# Average Stator Temperature
# --------------------------------------------------------

df["avg_stator_temp"] = (
    df["stator_winding"] +
    df["stator_tooth"] +
    df["stator_yoke"]
) / 3.0

# --------------------------------------------------------
# Temperature Difference
# --------------------------------------------------------

df["stator_coolant_delta"] = (
    df["avg_stator_temp"] -
    df["coolant"]
)

# --------------------------------------------------------
# PM Temperature Difference
# --------------------------------------------------------

df["pm_ambient_delta"] = (
    df["pm"] -
    df["ambient"]
)

# --------------------------------------------------------
# Torque Constant Estimate
# --------------------------------------------------------

epsilon = 1e-6

df["torque_constant"] = (
    df["torque"] /
    (df["current_mag"] + epsilon)
)

# --------------------------------------------------------
# Current Vector Angle
# --------------------------------------------------------

df["current_angle"] = np.arctan2(
    df["i_q"],
    df["i_d"]
)

# --------------------------------------------------------
# Voltage Vector Angle
# --------------------------------------------------------

df["voltage_angle"] = np.arctan2(
    df["u_q"],
    df["u_d"]
)

# --------------------------------------------------------
# Estimated Efficiency Proxy
# --------------------------------------------------------

df["efficiency_proxy"] = (
    df["electrical_power"] /
    (df["electrical_power"] +
     df["copper_loss"] +
     epsilon)
)

# --------------------------------------------------------
# Display New Features
# --------------------------------------------------------

print("\nNew Features Added")

new_features = [
    "current_mag",
    "voltage_mag",
    "electrical_power",
    "apparent_power",
    "copper_loss",
    "avg_stator_temp",
    "stator_coolant_delta",
    "pm_ambient_delta",
    "torque_constant",
    "current_angle",
    "voltage_angle",
    "efficiency_proxy"
]

for feature in new_features:
    print(feature)

print("\nTotal Features:", len(df.columns))

print("\nCurrent Dataset Shape:", df.shape)

STEP 3 : PHYSICS FEATURE ENGINEERING

New Features Added
current_mag
voltage_mag
electrical_power
apparent_power
copper_loss
avg_stator_temp
stator_coolant_delta
pm_ambient_delta
torque_constant
current_angle
voltage_angle
efficiency_proxy

Total Features: 25

Current Dataset Shape: (1330816, 25)


In [17]:
# ==========================================================
# STEP 4 - CREATE TIME SERIES WINDOWS
# ==========================================================

import numpy as np

WINDOW_SIZE = 64

TARGET_COLUMNS = [
    "pm",
    "stator_winding",
    "stator_tooth",
    "stator_yoke"
]

FEATURE_COLUMNS = [
    c for c in df.columns
    if c not in TARGET_COLUMNS + ["profile_id"]
]

print("="*60)
print("STEP 4 : SEQUENCE GENERATION")
print("="*60)

X = []
Y = []

profiles = df["profile_id"].unique()

print(f"Total Profiles : {len(profiles)}")

for profile in profiles:

    profile_df = df[df["profile_id"] == profile]

    feature_data = profile_df[FEATURE_COLUMNS].values
    target_data = profile_df[TARGET_COLUMNS].values

    for i in range(len(profile_df) - WINDOW_SIZE):

        X.append(feature_data[i:i+WINDOW_SIZE])

        Y.append(target_data[i+WINDOW_SIZE])

X = np.asarray(X, dtype=np.float32)
Y = np.asarray(Y, dtype=np.float32)

print("\nSequence Creation Complete")

print(f"Input Shape  : {X.shape}")
print(f"Target Shape : {Y.shape}")

print("\nFeature Count :", X.shape[2])

print("Window Length :", X.shape[1])

STEP 4 : SEQUENCE GENERATION
Total Profiles : 69

Sequence Creation Complete
Input Shape  : (1326400, 64, 20)
Target Shape : (1326400, 4)

Feature Count : 20
Window Length : 64
